# Figuras del paper: Experimentos A/B y warm start

Este notebook **sólo lee resultados terminados**. No ejecuta HiGHS, no genera redes y no cambia los experimentos.

Las corridas publicadas usan el objetivo **primary**, 150 instancias por experimento y tres inicializaciones: Cold, Benefit start y Density start. La calidad principal usa instancias certificadas; las figuras de presupuesto y warm start conservan la cohorte completa.

Datos fijados a `2cd8035c8417b3c4f5107c57ee174beef046ced3`. Las convenciones se explican en `TTAP_WarmStart/README_PAPER_FIGURES_ES.md`.


In [ ]:
# Sólo visualización: no importa HiGHS ni cambia el runtime del experimento.
from pathlib import Path
from urllib.request import urlopen
import hashlib, importlib.util, sys

PUBLICATION_REVISION = '7cbb2b3636f4daba115dbce906f0359aa6c32566'
PUBLICATION_SHA256 = '2a36a34b0d5e1c87eeb8a3adc257780a21eebd6e454356d705025c0cf55996f6'
_publication_name = 'paper_publication_figures.py'
_publication_candidates = [
    Path.cwd() / _publication_name,
    Path.cwd() / 'TTAP_WarmStart' / _publication_name,
    Path.cwd() / 'Resuelto_Paper' / 'TTAP_WarmStart' / _publication_name,
    Path.cwd().parent / 'TTAP_WarmStart' / _publication_name,
]
_publication_bytes = next((p.read_bytes() for p in _publication_candidates
    if p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest() == PUBLICATION_SHA256), None)
if _publication_bytes is None:
    _publication_url = ('https://raw.githubusercontent.com/educarrascov/DISC_TESIS/'
        + PUBLICATION_REVISION + '/Resuelto_Paper/TTAP_WarmStart/' + _publication_name)
    with urlopen(_publication_url, timeout=60) as response:
        _publication_bytes = response.read()
if hashlib.sha256(_publication_bytes).hexdigest() != PUBLICATION_SHA256:
    raise RuntimeError('La visualización descargada no coincide con la revisión publicada.')
_publication_folder = Path.cwd() / 'ttap_publication_helpers'
_publication_folder.mkdir(exist_ok=True)
_publication_path = _publication_folder / ('paper_publication_' + PUBLICATION_SHA256[:12] + '.py')
_publication_path.write_bytes(_publication_bytes)
_publication_spec = importlib.util.spec_from_file_location('ttap_paper_figures_' + PUBLICATION_SHA256[:12], _publication_path)
ttap_figures = importlib.util.module_from_spec(_publication_spec)
sys.modules[_publication_spec.name] = ttap_figures
_publication_spec.loader.exec_module(ttap_figures)
print('Visualización cargada: sólo lectura de resultados, sin optimizaciones.')


## 1. Estética y carpetas

Aquí se pueden cambiar colores, fuentes, límites y resolución. Se recuperan los lienzos, bandas y ejes de los notebooks ANYTIME. No se suavizan los datos ni se inventan puntos intermedios.

Para Overleaf usa los **PDF vectoriales**. El DPI afecta al PNG: 600 es el valor inicial; 2000 reproduce la exportación raster antigua y requiere más memoria. Cambiarlo no exige repetir las optimizaciones.


In [ ]:
STYLE = ttap_figures.style_config(png_dpi=600)
# STYLE['png_dpi'] = 2000
# STYLE['colors'].update({'Cold': '#1f77b4', 'GBStart': '#9467bd', 'GDStart': '#17becf'})
# STYLE['colors'].update({'A': '#0072B2', 'B': '#D55E00'})
# STYLE['rc']['font.size'] = 10
# STYLE['sizes']['quality'] = (9.0, 5.0)
# STYLE['available_ylim'] = (0, 0.65)  # Misma escala en ambos paneles A14/B14.

QUALITY_COHORT = 'Cold'  # También 'common' o 'reference'; se emparejan los greedy.
CERTIFICATION_TARGETS = (80, 95)
FRONTIER_METHODS = ('Cold', 'GBStart', 'GDStart')

# None descarga los CSV publicados. Alternativa: carpetas locales con
# manifest.json, metrics.csv y trace.csv de las corridas completas.
LOCAL_A = None  # Path('/ruta/ttap_runs_paper_2000dpi/A_paper_full_997dc6f9b016')
LOCAL_B = None  # Path('/ruta/ttap_runs_paper_2000dpi/B_paper_full_a1dfd983cde0')
PUBLISHED_CACHE = Path.cwd() / 'ttap_publication_data'
OUTPUT_DIR = Path.cwd() / 'ttap_publication_figures'


## 2. Leer y verificar los resultados

Se exige la grilla completa, el mismo modelo entre inicializaciones, un único objetivo y los dos greedy emparejados. Las descargas verifican los archivos de la revisión publicada. Las carpetas locales se validan con su propio manifest.


In [ ]:
from IPython.display import display, Image
A_DIR = Path(LOCAL_A) if LOCAL_A is not None else ttap_figures.download_published('A', PUBLISHED_CACHE)
B_DIR = Path(LOCAL_B) if LOCAL_B is not None else ttap_figures.download_published('B', PUBLISHED_CACHE)
A = ttap_figures.load_run(A_DIR, 'A')
B = ttap_figures.load_run(B_DIR, 'B')
for study in [A, B]:
    print(study['experiment'], '| instancias:', study['n'], '| corridas HiGHS:', len(study['warm']),
          '| objetivo:', study['manifest']['config']['objective']['mode'],
          '| óptimos conocidos:', len(study['references']))


## 3. Regenerar las secciones 4.1 y 4.2

A1 y B1 incluyen únicamente soluciones certificadas. Ambos greedy se evalúan en **esas mismas instancias**. Cuando hay menos de 30 certificaciones se usan puntos huecos con SD: la muestra es condicional.

Runtime y esfuerzo computacional conservan todos los casos. Filtrarlos por certificación ocultaría las búsquedas más difíciles.


In [ ]:
A_REPORT = ttap_figures.replot_run(A_DIR, 'A', OUTPUT_DIR / 'A', style=STYLE,
    quality_cohort=QUALITY_COHORT, include_budget_figures=True)
B_REPORT = ttap_figures.replot_run(B_DIR, 'B', OUTPUT_DIR / 'B', style=STYLE,
    quality_cohort=QUALITY_COHORT, include_budget_figures=True)
for report in [A_REPORT, B_REPORT]:
    print(report['directory'])
    display(report['coverage'])
    display(Image(filename=str(report['figures']['quality']['preview'])))


## 4. Comparación A/B y cuadros para Overleaf

`AB14_observed_available_benefit` une el panel central de A14 (izquierda) con el de B14 (derecha): beneficio disponible medio sobre los 150 casos. Sólo usa callbacks registrados hasta cada instante de una búsqueda larga. Incluye el greedy validado desde tiempo cero y idle=0 para Cold cuando no hay incumbent observado.

No es una media condicionada a tener incumbent ni una muestra filtrada por certificación futura. También se generan las fronteras de certificación con umbrales de 80 % y 95 %, un resumen de presupuesto y la comparación con cold + el mismo greedy de respaldo.


In [ ]:
COMPARISON = ttap_figures.combined_figures(A, B, OUTPUT_DIR / 'AB', style=STYLE,
    thresholds=CERTIFICATION_TARGETS, frontier_methods=FRONTIER_METHODS,
    include_attribution=True)
for name in ['warm_effect', 'observed_frontier_Cold', 'dashboard', 'injection']:
    if name in COMPARISON['figures']:
        print(name)
        display(Image(filename=str(COMPARISON['figures'][name]['preview'])))


## 5. Límite configurado y tiempo observado

Algunos jobs de B sobrepasan ampliamente el límite solicitado. Se separan dos lecturas:

- `AB5_observed_certification_*`: certificados observados dentro de la búsqueda larga a cada tiempo real. Es la recomendación para discutir presupuesto de solve time.
- `AB5_budget_certification_*`: resultados finales de reinicios independientes por límite configurado, aunque hayan terminado después.

No se recortan tiempos ni se incorporan resultados finales a checkpoints anteriores. El reloj observado excluye la preparación del modelo y del inicio; no equivale a un plazo operativo completo. El código no atribuye una causa al sobrepaso sin un diagnóstico adicional.


In [ ]:
import pandas as pd
for study in [A, B]:
    late = study['warm'].copy()
    late['Overrun ratio'] = late['Solver Time sec'] / late['Time Limit sec']
    print('Mayores sobrepasos de', study['experiment'])
    display(late.nlargest(6, 'Overrun ratio')[['Scenario', 'Method', 'Time Limit sec',
        'Solver Time sec', 'HiGHS Internal Time sec', 'Optimal Certified']])
display(COMPARISON['budget_status'][['Experiment', 'Start Method', 'Time Limit sec',
    'certified', 'feasible_uncertified', 'no_incumbent', 'mean_incumbent',
    'mean_available', 'mean_gap_pct', 'gap_n']])
print('Primer checkpoint observado que alcanza cada umbral:')
display(COMPARISON['observed_thresholds'])


## 6. Copiar al paper

A y B contienen las tablas 1--4 en el formato de cinco columnas del manuscrito. La carpeta AB incluye:

- `Table_AB_observed_budget.tex`: checkpoints, coherentes con A14/B14 combinada.
- `Table_AB_budget_status.tex`: resultados finales por límite configurado; categorías de estado, medias y denominadores.
- `Table_AB_certification_thresholds.tex`: primer límite configurado probado que alcanza cada umbral.
- `Figures_AB_overleaf.tex`: figuras y captions propuestos. Copia sólo los bloques que elijas.

Propuesta: A1/B1 y tablas en 4.1/4.2; AB14 y tabla observada en la sección de presupuesto. La frontera observada puede reemplazar la figura 5. AB15 permite discutir la atribución del efecto, preferiblemente en el suplemento. No es necesario usar todas las imágenes.

Se requieren `booktabs`, `makecell` y `graphicx`. Las brechas sólo usan referencias certificadas del mismo modelo. Los casos sin óptimo conocido siguen incluidos en las métricas de disponibilidad.


In [ ]:
for folder in [A_REPORT['directory'], B_REPORT['directory'], COMPARISON['directory']]:
    for path in sorted(folder.glob('*.tex')):
        print(path)

# Descomenta el archivo que quieras copiar a Overleaf.
# print((COMPARISON['directory'] / 'Table_AB_observed_budget.tex').read_text())
# print((B_REPORT['directory'] / 'Table_3_Experiment_B_quality.tex').read_text())
